<style>
.markdown-content {
    direction: rtl;
    text-align: right;
}

.markdown-content pre {
    direction: ltr;
    text-align: left;
}

.markdown-content code {
    direction: ltr;
    text-align: left;
}
</style>

<div class="markdown-content">

در این مثال بدون استفاده از Framework خاصی، منطق Parent-Child Retrieval را پیاده‌سازی می‌کنیم.
- Parentها متن‌های بزرگ‌تر هستند.
- Childها از تقسیم Parentها به جمله‌های کوچک‌تر ساخته می‌شوند.
- فقط Childها Embedding می‌شوند.
- بعد از جستجو، Parent مربوط به Childهای پیدا شده برگردانده می‌شود.

In [1]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Parent Documents
parents = [
    {
        "id": "P-001",
        "title": "Annual Leave Policy",
        "text": """
        Employees receive 26 annual leave days per year.
        Leave requests must be approved by the manager.
        Employees should submit their requests before the planned leave date.
        """
    },
    {
        "id": "P-002",
        "title": "Health Insurance Policy",
        "text": """
        Employees have supplementary health insurance.
        The insurance provider is selected by the HR department.
        Insurance coverage includes dental services.
        """
    },
    {
        "id": "P-003",
        "title": "Remote Work Policy",
        "text": """
        Employees can work remotely two days per week.
        Remote work requires manager approval.
        Employees must attend important office meetings.
        """
    }
]

# 2. Create Child Chunks
children = []

for parent in parents:

    sentences = [
        sentence.strip()
        for sentence in parent["text"].split(".")
        if sentence.strip()
    ]

    for index, sentence in enumerate(sentences):

        children.append({
            "id": f"{parent['id']}-C{index + 1}",
            "parent_id": parent["id"],
            "text": sentence
        })

# 3. Initialize Embedding Model
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

child_embeddings = model.encode(
    [child["text"] for child in children],
    convert_to_numpy=True,
    normalize_embeddings=True
)

# 4. Retrieve Relevant Children
def retrieve_children(query, top_k=3):

    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        query_embedding,
        child_embeddings
    )[0]

    indices = np.argsort(scores)[::-1][:top_k]

    return [
        {
            **children[index],
            "score": float(scores[index])
        }
        for index in indices
    ]


# 5. Retrieve Parent Documents
def retrieve_parents(retrieved_children):

    parent_ids = list(dict.fromkeys(
        child["parent_id"]
        for child in retrieved_children
    ))

    parent_lookup = {
        parent["id"]: parent
        for parent in parents
    }

    return [
        parent_lookup[parent_id]
        for parent_id in parent_ids
    ]


# 6. Execute Parent-Child Retrieval
query = "Who must approve an employee's annual leave request?"

# First: Search Child Chunks
retrieved_children = retrieve_children(
    query,
    top_k=3
)

# Second: Retrieve Corresponding Parents
retrieved_parents = retrieve_parents(
    retrieved_children
)

# 7. Display Results
print("RETRIEVED CHILD CHUNKS")

for child in retrieved_children:
    print("\nChild ID:", child["id"])
    print("Parent ID:", child["parent_id"])
    print("Text:", child["text"])
    print("Score:", round(child["score"], 4))

print("\n" + "=" * 60)
print("RETRIEVED PARENT DOCUMENTS")

for parent in retrieved_parents:
    print("\nParent ID:", parent["id"])
    print("Title:", parent["title"])
    print("Full Context:")
    print(parent["text"])

e:\AI-Engineering\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1314.29it/s]


RETRIEVED CHILD CHUNKS

Child ID: P-001-C2
Parent ID: P-001
Text: Leave requests must be approved by the manager
Score: 0.7689

Child ID: P-001-C3
Parent ID: P-001
Text: Employees should submit their requests before the planned leave date
Score: 0.7031

Child ID: P-001-C1
Parent ID: P-001
Text: Employees receive 26 annual leave days per year
Score: 0.5844

RETRIEVED PARENT DOCUMENTS

Parent ID: P-001
Title: Annual Leave Policy
Full Context:

        Employees receive 26 annual leave days per year.
        Leave requests must be approved by the manager.
        Employees should submit their requests before the planned leave date.
        
